In [ ]:
# ─────────────────────────────────────────────
# CELL 1: Import & Kurulum
# ─────────────────────────────────────────────
import os, copy, time, random, math, gc, warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import EfficientNet_B3_Weights
from PIL import Image
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support
)
from pathlib import Path
warnings.filterwarnings('ignore')

try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

print("Tüm kütüphaneler yüklendi.")

In [ ]:
# ─────────────────────────────────────────────
# CELL 2: HİPERPARAMETRELER
# ─────────────────────────────────────────────

# !! SADECE BU 2 SATIRI KENDİ YOLUNLA GÜNCELLE !!
TRAIN_DIR = r"C:\Users\Ali Çelik\Desktop\train"
TEST_DIR  = r"C:\Users\Ali Çelik\Desktop\test"

# ── Sabit parametreler ──
IMG_SIZE    = 300
BATCH_SIZE  = 16
EPOCHS      = 100       # Aşama 1
FT_EPOCHS   = 80        # Aşama 2 (artırıldı — daha çok fine-tune fırsatı)
SEED        = 42
VAL_SPLIT   = 0.15
NUM_WORKERS = 0          # Windows zorunlu

# ── Optimizasyon ──
LR            = 5e-4     # 0.0005 — sabit
FINE_TUNE_LR  = 2e-4     # Aşama 2 classifier lr (ESKİ: 3e-5 → ÇOK DÜŞÜKTÜ)
WARMUP_EPOCHS = 3        # Daha kısa warmup
PATIENCE_S1   = 15       # Aşama 1 early stopping
PATIENCE_S2   = 15       # Aşama 2 early stopping
LABEL_SMOOTH  = 0.1
GRAD_CLIP     = 1.0
TTA_N         = 5

MODEL_SAVE_PATH = "best_model_s1.pth"
FT_SAVE_PATH    = "best_model_finetuned.pth"

# ── 6 Hedef Sınıf ──
TARGET_CLASSES = [
    "Acne and Rosacea Photos",
    "Eczema Photos",
    "Psoriasis pictures Lichen Planus and related diseases",
    "Urticaria Hives",
    "Nail Fungus and other Nail Disease",
    "Scabies Lyme Disease and other Infestations",
]
SHORT_NAMES = ["Acne", "Eczema", "Psoriasis", "Urticaria", "Nail", "Scabies"]

print(f"Model      : EfficientNet-B3 V3")
print(f"Img size   : {IMG_SIZE}")
print(f"Batch size : {BATCH_SIZE}")
print(f"LR         : {LR}")
print(f"FT LR      : {FINE_TUNE_LR}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 3: Seed & Device
# ─────────────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark    = True
        torch.backends.cudnn.deterministic = False

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
else:
    print("CPU modu — eğitim yavaş olacak!")

In [ ]:
# ─────────────────────────────────────────────
# CELL 4: Dataset Sınıfı
# ─────────────────────────────────────────────
# V3 DEĞİŞİKLİK: ROI extraction KALDIRILDI.
# Neden: Otsu thresholding dermatoloji görüntülerinde güvenilir değil,
# yanlış crop model performansını ciddi şekilde düşürüyordu.
# Şimdi düz resize + augmentation ile çalışıyoruz.

import cv2

class SkinDataset(Dataset):
    """Basit, güvenilir dataset — ROI yok, sadece resize."""
    def __init__(self, root_dir, transform=None, target_size=300,
                 target_classes=None):
        self.root_dir    = root_dir
        self.transform   = transform
        self.target_size = target_size

        if target_classes is not None:
            self.classes = sorted([
                d for d in target_classes
                if os.path.isdir(os.path.join(root_dir, d))
            ])
        else:
            self.classes = sorted([
                d for d in os.listdir(root_dir)
                if os.path.isdir(os.path.join(root_dir, d))
            ])

        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}

        VALID_EXT = (".jpg", ".jpeg", ".png", ".bmp")
        self.samples = []
        for cls in self.classes:
            cls_dir = os.path.join(root_dir, cls)
            for fname in os.listdir(cls_dir):
                if os.path.splitext(fname)[1].lower() in VALID_EXT:
                    fpath = os.path.join(cls_dir, fname)
                    self.samples.append((fpath, self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            # Türkçe karakter desteği
            raw = np.fromfile(img_path, dtype=np.uint8)
            img = cv2.imdecode(raw, cv2.IMREAD_COLOR)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (self.target_size, self.target_size))
        except:
            img = np.zeros((self.target_size, self.target_size, 3), dtype=np.uint8)

        pil_img = Image.fromarray(img)
        if self.transform:
            pil_img = self.transform(pil_img)
        return pil_img, label


class TransformSubset(Dataset):
    """random_split subset'ine farklı transform uygular."""
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


print("Dataset sınıfları tanımlandı (ROI kaldırıldı).")

In [ ]:
# ─────────────────────────────────────────────
# CELL 5: Transform & DataLoader
# ─────────────────────────────────────────────
# V3 DEĞİŞİKLİK: Augmentation ciddi şekilde sadeleştirildi.
# Eski: 8+ transform birbirine zincirleniyor, AutoAugment dahil
#   → görüntü tanınmaz hale geliyordu, model öğrenemiyordu
# Yeni: Dermatolojiye uygun hafif augmentation
#   → renk/doku bilgisi korunuyor, yeterli varyasyon sağlanıyor

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),  # biraz büyük yükle
    transforms.RandomCrop(IMG_SIZE),                     # rastgele kırp
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.02),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), shear=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.15)),  # hafif erasing
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

# TTA transforms
tta_transforms = [
    eval_transform,
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)
    ]),
    transforms.Compose([
        transforms.Resize((int(IMG_SIZE * 1.1), int(IMG_SIZE * 1.1))),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)
    ]),
    transforms.Compose([
        transforms.Resize((int(IMG_SIZE * 1.15), int(IMG_SIZE * 1.15))),
        transforms.FiveCrop(IMG_SIZE),
        transforms.Lambda(lambda crops: crops[0]),  # center-ish crop
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)
    ]),
]

# ── Veri yükle ──
print("Veri seti yükleniyor...")
full_train_dataset = SkinDataset(
    TRAIN_DIR, transform=None,
    target_size=IMG_SIZE, target_classes=TARGET_CLASSES
)
class_names = full_train_dataset.classes
NUM_CLASSES = len(class_names)
print(f"Bulunan sınıflar ({NUM_CLASSES}): {class_names}")

train_size = int((1 - VAL_SPLIT) * len(full_train_dataset))
val_size   = len(full_train_dataset) - train_size
train_subset, val_subset = random_split(
    full_train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_dataset = TransformSubset(train_subset, train_transform)
val_dataset   = TransformSubset(val_subset,   eval_transform)
test_dataset  = SkinDataset(
    TEST_DIR, transform=eval_transform,
    target_size=IMG_SIZE, target_classes=TARGET_CLASSES
)

# ── WeightedRandomSampler ──
labels_all    = [full_train_dataset.samples[i][1] for i in train_subset.indices]
class_counts  = np.bincount(labels_all, minlength=NUM_CLASSES)
class_weights_np = 1.0 / (class_counts + 1e-8)
sample_weights   = [class_weights_np[l] for l in labels_all]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Loss ağırlıkları — sqrt normalizasyonu (aşırı ağırlıklandırmayı önler)
total_samples   = len(labels_all)
loss_weights    = total_samples / (NUM_CLASSES * class_counts)
loss_weights    = np.sqrt(loss_weights)  # sqrt ile yumuşat
loss_weights    = loss_weights / loss_weights.sum() * NUM_CLASSES  # normalize
class_weights_t = torch.tensor(loss_weights, dtype=torch.float32).to(device)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    sampler=sampler, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'), drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda')
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda')
)

print(f"\nTrain: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}")
print("\nSınıf dağılımı (train):")
for i, cls in enumerate(class_names):
    c = (np.array(labels_all) == i).sum()
    print(f"  [{i}] {SHORT_NAMES[i]:<12} {c:4d} görüntü  (loss_w={loss_weights[i]:.3f})")

In [ ]:
# ─────────────────────────────────────────────
# CELL 6: Model — EfficientNet-B3
# ─────────────────────────────────────────────
# V3 DEĞİŞİKLİK: Classifier head sadeleştirildi.
# Eski: Dropout(0.4) → Linear(1536,512) → BN → ReLU → Dropout(0.3) → Linear(512,6)
#   → Backbone donukken bu kadar parametre aşırı, overfit veya underfit
# Yeni: Dropout(0.3) → Linear(1536, 256) → ReLU → Dropout(0.2) → Linear(256, 6)
#   → Daha dengeli, hızlı converge

model = models.efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)

# Aşama 1: Backbone tamamen dondur
for param in model.features.parameters():
    param.requires_grad = False

in_features = model.classifier[1].in_features  # 1536

model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.2),
    nn.Linear(256, NUM_CLASSES)
)

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"EfficientNet-B3 | in_features={in_features}")
print(f"Toplam parametre      : {total_p:,}")
print(f"Eğitilebilir (Aş. 1) : {trainable:,}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 7: Loss, Scheduler, Eğitim Fonksiyonları
# ─────────────────────────────────────────────
# V3 DEĞİŞİKLİK: Focal Loss KALDIRILDI → CrossEntropy + label smoothing.
# Neden: Focal Loss + class weights + mixup + sampler hepsi birden
# dengesizliği düzeltmeye çalışıyordu → çelişkili sinyaller.
# Şimdi: WeightedRandomSampler + Label Smoothing CE yeterli.
# Mixup da kaldırıldı — backbone donukken fayda yerine zarar veriyor.

# ── Warmup + Cosine Scheduler ──
def get_warmup_cosine_scheduler(optimizer, warmup_epochs, total_epochs):
    def lr_lambda(current_epoch):
        if current_epoch < warmup_epochs:
            return float(current_epoch + 1) / float(max(1, warmup_epochs))
        progress = float(current_epoch - warmup_epochs) / float(
            max(1, total_epochs - warmup_epochs))
        return max(0.01, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ── Eğitim fonksiyonu (temiz, mixup yok) ──
def train_one_epoch(model, loader, criterion, optimizer, device, grad_clip=1.0):
    model.train()
    running_loss = correct = total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    return running_loss / total, correct / total


# ── Mixup (sadece Aşama 2'de kullanılacak) ──
def mixup_data(x, y, alpha=0.2, device='cpu'):
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)  # dominant label her zaman > 0.5
    idx = torch.randperm(x.size(0)).to(device)
    mixed = lam * x + (1 - lam) * x[idx]
    return mixed, y, y[idx], lam


def train_one_epoch_mixup(model, loader, criterion, optimizer, device,
                          mixup_alpha=0.2, grad_clip=1.0):
    model.train()
    running_loss = correct = total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        images, y_a, y_b, lam = mixup_data(images, labels, mixup_alpha, device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == y_a).sum().item()
        total   += labels.size(0)

    return running_loss / total, correct / total


# ── Değerlendirme ──
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = correct = total = 0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(images)
            probs   = torch.softmax(outputs, dim=1)
            loss    = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    acc = correct / total
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0)
    return (running_loss / total, acc, prec, rec, f1,
            np.array(all_labels), np.array(all_preds), np.array(all_probs))


# ── TTA ──
def evaluate_with_tta(model, test_dir, device, n_tta=5):
    model.eval()
    first_ds = SkinDataset(test_dir, transform=eval_transform,
                           target_size=IMG_SIZE, target_classes=TARGET_CLASSES)
    all_labels = [s[1] for s in first_ds.samples]
    n_samples  = len(all_labels)
    sum_probs  = np.zeros((n_samples, NUM_CLASSES))

    for t_idx, tta_tf in enumerate(tta_transforms[:n_tta]):
        tta_ds     = SkinDataset(test_dir, transform=tta_tf,
                                 target_size=IMG_SIZE, target_classes=TARGET_CLASSES)
        tta_loader = DataLoader(tta_ds, batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=NUM_WORKERS)
        batch_probs = []
        with torch.no_grad():
            for images, _ in tta_loader:
                images = images.to(device)
                probs  = torch.softmax(model(images), dim=1)
                batch_probs.append(probs.cpu().numpy())
        sum_probs += np.concatenate(batch_probs, axis=0)
        print(f"  TTA {t_idx+1}/{n_tta} tamamlandı")

    avg_probs  = sum_probs / n_tta
    y_pred_tta = np.argmax(avg_probs, axis=1)
    y_true     = np.array(all_labels)
    acc        = (y_pred_tta == y_true).mean()
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred_tta, average="macro", zero_division=0)
    return acc, prec, rec, f1, y_true, y_pred_tta


print("Eğitim araçları hazır.")

In [ ]:
# ─────────────────────────────────────────────
# CELL 8: AŞAMA 1 — Classifier Eğitimi
# ─────────────────────────────────────────────
# V3: Sade CrossEntropy + Label Smoothing, Mixup yok
# Backbone donukken classifier'ı hızlı ve kararlı eğit.

print("=" * 55)
print("AŞAMA 1: CLASSIFIER EĞİTİMİ (Backbone donuk)")
print("=" * 55)

criterion = nn.CrossEntropyLoss(
    weight=class_weights_t,
    label_smoothing=LABEL_SMOOTH
)

optimizer = optim.AdamW(
    model.classifier.parameters(),
    lr=LR,
    weight_decay=1e-2
)

scheduler = get_warmup_cosine_scheduler(
    optimizer, warmup_epochs=WARMUP_EPOCHS, total_epochs=EPOCHS)

best_val_acc   = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
no_improve     = 0
history        = []

for epoch in range(EPOCHS):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device, grad_clip=GRAD_CLIP
    )

    val_loss, val_acc, val_prec, val_rec, val_f1, _, _, _ = evaluate(
        model, val_loader, criterion, device)
    scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']
    history.append({'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss,   'val_acc': val_acc})

    print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Val Acc: {val_acc:.4f} F1: {val_f1:.4f} | "
          f"LR: {current_lr:.2e} | {time.time()-t0:.0f}s")

    if val_acc > best_val_acc:
        best_val_acc   = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        no_improve = 0
        print(f"   Model kaydedildi (best val_acc={best_val_acc:.4f})")
    else:
        no_improve += 1
        if no_improve >= PATIENCE_S1:
            print(f"Early stopping! ({PATIENCE_S1} epoch iyileşme yok)")
            break

model.load_state_dict(best_model_wts)
_, phase1_acc, _, _, phase1_f1, _, _, _ = evaluate(
    model, test_loader, criterion, device)
print(f"\nAşama 1 Test Accuracy: {phase1_acc:.4f}  F1: {phase1_f1:.4f}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 9: AŞAMA 2 — Fine-Tuning
# ─────────────────────────────────────────────
# V3 DEĞİŞİKLİK:
#   1. Bloklar 3-7 açılıyor (0,1,2 donuk — erken katmanlar genel)
#   2. LR'ler ÇOOOK artırıldı: eski 1e-6/3e-6/3e-5 → yeni 1e-5/5e-5/2e-4
#   3. Mixup SADECE burada (alpha=0.2, hafif)
#   4. 80 epoch (eskiden 60)

print("=" * 55)
print("AŞAMA 2: FINE-TUNING")
print("=" * 55)

# Blokları aç
UNFREEZE_BLOCKS = [3, 4, 5, 6, 7]
for name, param in model.features.named_parameters():
    if any(name.startswith(str(idx) + ".") for idx in UNFREEZE_BLOCKS):
        param.requires_grad = True

trainable_ft = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Açılan bloklar: {UNFREEZE_BLOCKS}")
print(f"Eğitilebilir parametre: {trainable_ft:,}")

# Differential LR — her grup için MAKUL değerler
mid_params  = [p for n, p in model.features.named_parameters()
               if p.requires_grad and any(
                   n.startswith(str(i) + ".") for i in [3, 4])]
late_params = [p for n, p in model.features.named_parameters()
               if p.requires_grad and any(
                   n.startswith(str(i) + ".") for i in [5, 6, 7])]
head_params = list(model.classifier.parameters())

optimizer_ft = optim.AdamW([
    {"params": mid_params,  "lr": FINE_TUNE_LR / 20},   # 1e-5
    {"params": late_params, "lr": FINE_TUNE_LR / 4},    # 5e-5
    {"params": head_params, "lr": FINE_TUNE_LR},         # 2e-4
], weight_decay=1e-3)

scheduler_ft = get_warmup_cosine_scheduler(
    optimizer_ft, warmup_epochs=3, total_epochs=FT_EPOCHS)

# Fine-tune criterion — sınıf ağırlıkları olmadan (sampler zaten dengeliyor)
criterion_ft = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

best_ft_acc = 0.0
best_ft_wts = copy.deepcopy(model.state_dict())
ft_no_imp   = 0

for epoch in range(FT_EPOCHS):
    t0 = time.time()

    # İlk 5 epoch mixup yok — stabilize olsun; sonra hafif mixup
    if epoch < 5:
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion_ft, optimizer_ft, device,
            grad_clip=GRAD_CLIP
        )
    else:
        train_loss, train_acc = train_one_epoch_mixup(
            model, train_loader, criterion_ft, optimizer_ft, device,
            mixup_alpha=0.2, grad_clip=GRAD_CLIP
        )

    val_loss, val_acc, _, _, val_f1, _, _, _ = evaluate(
        model, val_loader, criterion_ft, device)
    scheduler_ft.step()

    current_lr = optimizer_ft.param_groups[-1]['lr']
    print(f"[FT] Epoch {epoch+1:3d}/{FT_EPOCHS} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Val Acc: {val_acc:.4f} F1: {val_f1:.4f} | "
          f"LR: {current_lr:.2e} | {time.time()-t0:.0f}s")

    if val_acc > best_ft_acc:
        best_ft_acc = val_acc
        best_ft_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), FT_SAVE_PATH)
        ft_no_imp = 0
        print(f"   Fine-tuned model kaydedildi ({best_ft_acc:.4f})")
    else:
        ft_no_imp += 1
        if ft_no_imp >= PATIENCE_S2:
            print(f"\n{PATIENCE_S2} epoch gelişme yok → erken durdurma.")
            break

model.load_state_dict(best_ft_wts)
print(f"\nFine-tuning tamamlandı. En iyi Val Acc: {best_ft_acc:.4f}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 10: AŞAMA 3 — Tüm Model Full Fine-Tuning (YENİ!)
# ─────────────────────────────────────────────
# V3 YENİ AŞAMA: Tüm backbone'u çok düşük LR ile açıp
# son bir tur daha eğitim. Bu, son %5-10 accuracy kazanımını sağlar.

print("=" * 55)
print("AŞAMA 3: FULL FINE-TUNING (Tüm model açık)")
print("=" * 55)

# TÜM parametreleri aç
for param in model.parameters():
    param.requires_grad = True

trainable_all = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Eğitilebilir parametre: {trainable_all:,}")

FULL_FT_EPOCHS = 30
FULL_FT_LR     = 5e-5   # çok düşük lr — pretrained ağırlıkları bozmamak için

# Stem (0,1) çok düşük, mid (2,3,4) düşük, late (5,6,7) orta, head yüksek
stem_params = [p for n, p in model.features.named_parameters()
               if any(n.startswith(str(i) + ".") for i in [0, 1, 2])]
mid_params2 = [p for n, p in model.features.named_parameters()
               if any(n.startswith(str(i) + ".") for i in [3, 4])]
late_params2 = [p for n, p in model.features.named_parameters()
                if any(n.startswith(str(i) + ".") for i in [5, 6, 7])]
head_params2 = list(model.classifier.parameters())

optimizer_full = optim.AdamW([
    {"params": stem_params,  "lr": FULL_FT_LR / 50},   # 1e-6
    {"params": mid_params2,  "lr": FULL_FT_LR / 5},    # 1e-5
    {"params": late_params2, "lr": FULL_FT_LR},          # 5e-5
    {"params": head_params2, "lr": FULL_FT_LR * 3},     # 1.5e-4
], weight_decay=1e-3)

scheduler_full = get_warmup_cosine_scheduler(
    optimizer_full, warmup_epochs=2, total_epochs=FULL_FT_EPOCHS)

criterion_full = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

best_full_acc = best_ft_acc
best_full_wts = copy.deepcopy(model.state_dict())
full_no_imp   = 0

for epoch in range(FULL_FT_EPOCHS):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch_mixup(
        model, train_loader, criterion_full, optimizer_full, device,
        mixup_alpha=0.15, grad_clip=GRAD_CLIP
    )

    val_loss, val_acc, _, _, val_f1, _, _, _ = evaluate(
        model, val_loader, criterion_full, device)
    scheduler_full.step()

    current_lr = optimizer_full.param_groups[-1]['lr']
    print(f"[FULL] Epoch {epoch+1:3d}/{FULL_FT_EPOCHS} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Val Acc: {val_acc:.4f} F1: {val_f1:.4f} | "
          f"LR: {current_lr:.2e} | {time.time()-t0:.0f}s")

    if val_acc > best_full_acc:
        best_full_acc = val_acc
        best_full_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), "best_model_full.pth")
        full_no_imp = 0
        print(f"   Full model kaydedildi ({best_full_acc:.4f})")
    else:
        full_no_imp += 1
        if full_no_imp >= 10:
            print(f"\n10 epoch gelişme yok → erken durdurma.")
            break

model.load_state_dict(best_full_wts)
print(f"\nAşama 3 tamamlandı. En iyi Val Acc: {best_full_acc:.4f}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 11: Test — Normal + TTA
# ─────────────────────────────────────────────

criterion_eval = nn.CrossEntropyLoss()

# ── Normal Değerlendirme ──
print("Normal değerlendirme yapılıyor...")
(test_loss, test_acc, test_prec, test_rec,
 test_f1, y_true, y_pred, y_probs) = evaluate(
    model, test_loader, criterion_eval, device)

print(f"\n{'='*55}")
print("TEST SONUÇLARI — Normal")
print(f"{'='*55}")
print(f"Test Accuracy  : {test_acc:.4f}")
print(f"Test F1 (macro): {test_f1:.4f}")

# ── TTA Değerlendirme ──
print(f"\nTTA değerlendirmesi başlıyor ({TTA_N} augmentation)...")
tta_acc, tta_prec, tta_rec, tta_f1, y_true_tta, y_pred_tta = evaluate_with_tta(
    model, TEST_DIR, device, n_tta=TTA_N)

print(f"\n{'='*55}")
print("TEST SONUÇLARI — TTA")
print(f"{'='*55}")
print(f"TTA Accuracy  : {tta_acc:.4f}")
print(f"TTA F1 (macro): {tta_f1:.4f}")
print(f"\n>>> Normal: {test_acc:.4f}  |  TTA: {tta_acc:.4f}  "
      f"(kazanım: +{(tta_acc-test_acc)*100:.2f}%)")

# En iyi sonucu kullan
if tta_acc > test_acc:
    final_y_true = y_true_tta
    final_y_pred = y_pred_tta
    final_acc    = tta_acc
    print("Nihai sonuç için TTA kullanılıyor.")
else:
    final_y_true = y_true
    final_y_pred = y_pred
    final_acc    = test_acc
    print("Nihai sonuç için normal değerlendirme kullanılıyor.")

# Sınıf bazlı rapor
short_map    = {full: short for full, short in zip(class_names, SHORT_NAMES[:NUM_CLASSES])}
target_names = [short_map.get(c, c.split()[0]) for c in class_names]

print("\nClassification Report (Nihai):")
print(classification_report(final_y_true, final_y_pred,
                            target_names=target_names, zero_division=0))

# Confusion Matrix
cm = confusion_matrix(final_y_true, final_y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
ax.set_title(f"Confusion Matrix — EfficientNet-B3 V3 (Acc: {final_acc:.4f})",
             fontsize=13)
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(target_names, rotation=45, ha="right", fontsize=10)
ax.set_yticklabels(target_names, fontsize=10)
ax.set_xlabel("Tahmin", fontsize=11)
ax.set_ylabel("Gerçek", fontsize=11)

thresh = cm.max() / 2.0
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=10,
                color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.savefig("confusion_matrix_v3.jpg", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nEğitim ve Test Tamamlandı! Final Accuracy: {final_acc:.4f}")